# OLED Display Interface and Text Rendering

This notebook demonstrates how to interface a **1.3-inch I2C OLED display** with the board using the **AXI IIC core** and **MMIO-based communication**. The OLED is driven directly through low-level I2C register access rather than a high-level driver.

The display is initialized using a sequence of **SSD1306-compatible commands**, configuring the correct orientation, addressing mode, and contrast settings. Text is rendered using a custom **8×8 bitmap font**, with glyphs rotated in software to ensure proper horizontal orientation on the screen.

This implementation allows full control over OLED behavior, including screen clearing, cursor positioning, and custom text rendering, making it suitable for real-time system feedback in the autonomous vehicle platform.

## References
- SSD1306 OLED Controller Datasheet  
- https://pynq.readthedocs.io/en/latest/  
- https://learn.adafruit.com/monochrome-oled-breakouts


In [9]:
#Run this One 
import time
from pynq import Overlay, MMIO

# -------------------------------
# 1. Load bitstream
# -------------------------------
ol = Overlay("car.bit", ignore_version=True)

# -------------------------------
# 2. MMIO for AXI I2C
# -------------------------------
iic = MMIO(ol.ip_dict["axi_iic_0"]["phys_addr"], 0x10000)

CR   = 0x100
SR   = 0x104
TX   = 0x108


# -------------------------------
# 3. AXI IIC low-level functions
# -------------------------------
def wait_tx_empty():
    while (iic.read(SR) & 0x80) == 0:
        pass

def iic_reset():
    iic.write(CR, 0x02)
    time.sleep(0.001)
    iic.write(CR, 0x01 | 0x04)   # Enable + Master
    while (iic.read(SR) & 0x02): # wait bus not busy
        pass

def iic_start_write(addr):
    wait_tx_empty()
    iic.write(TX, 0x100 | (addr << 1))

def iic_write_byte(b, stop=False):
    wait_tx_empty()
    if stop:
        b |= 0x200
    iic.write(TX, b)


# -------------------------------
# 4. OLED commands
# -------------------------------
OLED_ADDR = 0x3C

def oled_cmd(c):
    iic_start_write(OLED_ADDR)
    iic_write_byte(0x00)
    iic_write_byte(c, stop=True)
    time.sleep(0.001)

def oled_data(data):
    iic_start_write(OLED_ADDR)
    iic_write_byte(0x40)
    for i, b in enumerate(data):
        iic_write_byte(b, stop=(i == len(data)-1))
    time.sleep(0.001)


# -------------------------------
# 5. OLED INIT (correct orientation)
# -------------------------------
def oled_init():
    iic_reset()
    time.sleep(0.1)

    cmds = [
        0xAE,        # Display off
        0xD5, 0x80,  # Clock div
        0xA8, 0x3F,  # Mux ratio
        0xD3, 0x00,  # Offset
        0x40,        # Start line

        0x8D, 0x14,  # Charge pump
        0x20, 0x00,  # Horizontal addressing mode

        0xA1,        # Segment remap (OLED needed this)
        0xC0,        # COM scan direction

        0xDA, 0x12,  # COM pins hardware config
        0x81, 0xCF,  # Contrast
        0xD9, 0xF1,  # Precharge
        0xDB, 0x40,  # VCOMH
        0xA4,        # Display RAM
        0xA6,        # Normal display
        0xAF         # Display ON
    ]

    for c in cmds:
        oled_cmd(c)


# -------------------------------
# 6. Clear screen
# -------------------------------
def oled_clear():
    oled_cmd(0x21); oled_cmd(0); oled_cmd(127)
    oled_cmd(0x22); oled_cmd(0); oled_cmd(7)
    oled_data([0x00] * 1024)


# -------------------------------
# 7. 8x8 font
# -------------------------------
font = {
    'J': [0x0E,0x04,0x04,0x04,0x04,0x44,0x44,0x38],
    'E': [0x7E,0x40,0x40,0x7C,0x40,0x40,0x7E,0x00],
    'R': [0x7C,0x42,0x42,0x7C,0x48,0x44,0x42,0x00],
    'M': [0x42,0x66,0x5A,0x42,0x42,0x42,0x42,0x00],
    'I': [0x3E,0x08,0x08,0x08,0x08,0x08,0x3E,0x00],
    'A': [0x18,0x24,0x42,0x42,0x7E,0x42,0x42,0x00],
    'H': [0x42,0x42,0x42,0x7E,0x42,0x42,0x42,0x00],
}


# -------------------------------
# 8. Rotate glyph 90° clockwise ONLY
# -------------------------------
def rotate_glyph_90(glyph):
    rotated = [0]*8
    for row in range(8):
        for col in range(8):
            if glyph[row] & (1 << col):
                rotated[col] |= (1 << (7 - row))
    return rotated


# -------------------------------
# 9. Horizontal text rendering
# -------------------------------
def oled_text(x, y_page, text):
    col = x
    for ch in text.upper():
        glyph = font.get(ch, [0]*8)

        # Rotate ONLY — DO NOT MIRROR
        glyph = rotate_glyph_90(glyph)

        oled_cmd(0xB0 + y_page)
        oled_cmd(col & 0x0F)
        oled_cmd(0x10 | (col >> 4))
        oled_data(glyph)

        col += 9


# -------------------------------
# 10. RUN
# -------------------------------
oled_init()
oled_clear()
oled_text(20, 3, "JEREMIAH")

print("Horizontal text drawn correctly!")



Horizontal text drawn correctly!
